# NB03c2 -- PAHs: Bell-Shift, Alkylation & Source Fingerprints

Analyzes 22 alkylated PAHs (5 series) and 15 priority PAHs from ECCC ESTS.
Quantifies bell-shift, alkylation centroids, source ratios, and protocol validation.

**Depends on:** NB03c (molecular_fingerprint_stats with CPI/ACL/TAR)
**Produces:** PAH centroids, shift magnitudes, C0 depletion stats; ~13 figures

In [ ]:
# C01 -- Setup
import sys, warnings, re, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.stats import kruskal
from IPython.display import display, Image
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..') / 'notebooks'))
from utils import get_conn, STAGE_COLORS, OILTYPE_COLORS, setup_figure_style, FIG_ROOT

setup_figure_style()
import matplotlib as _mpl
_mpl.rcParams['font.family'] = 'DejaVu Sans'

FIG_DIR = FIG_ROOT / 'nb03c2'
if FIG_DIR.exists():
    shutil.rmtree(FIG_DIR)
FIG_DIR.mkdir(parents=True, exist_ok=True)

STAGE_ORDER = ['W0', 'W1', 'W2', 'W3']
STAGE_EVAP = {'W0': 0.00, 'W1': 0.10, 'W2': 0.20, 'W3': 0.30}
STAGE_LABELS = {'W0': 'W0 (fresh)', 'W1': 'W1 (~10%)', 'W2': 'W2 (~20%)', 'W3': 'W3 (~30%)'}
OIL_TYPES_ML = ['crude', 'bitumen_blend', 'refined', 'synthetic']

PAH_ABBREV = {
    'Naphthalene': 'Naph', 'Phenanthrene': 'Phe',
    'Dibenzothiophene': 'DBT', 'Fluorene': 'Flu', 'Chrysene': 'Chr',
}

def parse_pah(name):
    """Parse 'C2-Naphthalene' -> (2, 'Naphthalene'). Returns (None, None) on failure."""
    m = re.match(r'^C(\d)-(.+)$', name)
    if m:
        return int(m.group(1)), m.group(2)
    return None, None

print(f'Setup OK | FIG_DIR: {FIG_DIR}')

In [ ]:
# C02 -- Sanity check + DELETE selective
with get_conn() as conn:
    # Count PAH compounds by group
    checks = pd.read_sql('''
        SELECT compound_group, COUNT(*) AS n
        FROM compounds
        WHERE compound_group IN ('pah_alkylated', 'pah_priority')
          AND excluded = 0
        GROUP BY compound_group
    ''', conn)
    print('PAH compound inventory:')
    display(checks)

    # Series inventory with degrees
    series_inv = pd.read_sql('''
        SELECT compound_name FROM compounds
        WHERE compound_group = 'pah_alkylated' AND excluded = 0
        ORDER BY compound_name
    ''', conn)
    print('\nAlkylated PAH series inventory:')
    series_dict = {}
    for name in series_inv['compound_name']:
        deg, parent = parse_pah(name)
        if parent is not None:
            series_dict.setdefault(parent, []).append(deg)
    for parent, degrees in sorted(series_dict.items()):
        abbr = PAH_ABBREV.get(parent, parent[:4])
        print(f'  {parent} ({abbr}): C{min(degrees)}-C{max(degrees)} ({len(degrees)} members)')

    # DELETE existing pah_* stats for idempotency
    n_del = conn.execute("DELETE FROM molecular_fingerprint_stats WHERE stat_name LIKE 'pah_%'").rowcount
    print(f'\nDeleted {n_del} existing pah_* stats from molecular_fingerprint_stats.')

## PART 1 -- Bell-Shape Profiles

In [ ]:
# C03 -- Load PAH alkylated measurements
with get_conn() as conn:
    df_pah = pd.read_sql('''
        SELECT m.oil_id, o.oil_name, o.oil_type,
               m.stage_code AS stage, c.compound_name,
               m.value_imputed AS intensity
        FROM measurements m
        JOIN oils o ON m.oil_id = o.oil_id
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_group = 'pah_alkylated'
          AND c.excluded = 0 AND o.include_in_analysis = 1
          AND m.stage_code IN ('W0','W1','W2','W3')
    ''', conn)

# Parse series and alkyl degree
df_pah[['alkyl_degree', 'series']] = df_pah['compound_name'].apply(
    lambda x: pd.Series(parse_pah(x))
)
df_pah = df_pah.dropna(subset=['series']).copy()
df_pah['alkyl_degree'] = df_pah['alkyl_degree'].astype(int)

SERIES_ACTIVE = sorted(df_pah['series'].unique())
print(f'Loaded {len(df_pah)} PAH alkylated measurements')
print(f'Oils: {df_pah["oil_id"].nunique()}, Stages: {df_pah["stage"].nunique()}')
print(f'Active series ({len(SERIES_ACTIVE)}): {SERIES_ACTIVE}')
print(f'\nRecords per series:')
print(df_pah.groupby('series').size().to_string())

In [ ]:
# C04 -- F01 Bell-shape W0 all series x oil_type
n_series = len(SERIES_ACTIVE)
n_types = len(OIL_TYPES_ML)
fig, axes = plt.subplots(n_series, n_types, figsize=(4 * n_types, 3 * n_series),
                         constrained_layout=True)
if n_series == 1:
    axes = axes[np.newaxis, :]

sub = df_pah[df_pah['stage'] == 'W0'].copy()

for i, series in enumerate(SERIES_ACTIVE):
    ss = sub[sub['series'] == series].copy()
    # Normalize within each sample: relative intensity = intensity / sum(series)
    totals = ss.groupby('oil_id')['intensity'].transform('sum')
    ss['rel_intensity'] = ss['intensity'] / totals.replace(0, np.nan)
    degrees = sorted(ss['alkyl_degree'].unique())

    for j, ot in enumerate(OIL_TYPES_ML):
        ax = axes[i, j]
        sst = ss[ss['oil_type'] == ot]
        if sst.empty:
            ax.set_visible(False)
            continue
        medians = sst.groupby('alkyl_degree')['rel_intensity'].median()
        ax.bar([str(d) for d in degrees],
               [medians.get(d, 0) for d in degrees],
               color=OILTYPE_COLORS.get(ot, 'gray'), alpha=0.8)
        abbr = PAH_ABBREV.get(series, series[:4])
        if j == 0:
            ax.set_ylabel(f'{abbr}\nRelative intensity')
        if i == 0:
            ax.set_title(ot.replace('_', ' ').title())
        if i == n_series - 1:
            ax.set_xlabel('Alkylation degree')

fig.suptitle('F01 -- Bell-Shape Profiles at W0 by Series and Oil Type', fontsize=14, y=1.01)
fig_path = FIG_DIR / 'F01_bell_shape_W0_all_series.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

In [ ]:
# C05 -- F02 Bell-shift W0 vs W3 comparison all series (crude only)
crude = df_pah[df_pah['oil_type'] == 'crude'].copy()

fig, axes = plt.subplots(1, n_series, figsize=(4 * n_series, 4), constrained_layout=True)
if n_series == 1:
    axes = [axes]

for i, series in enumerate(SERIES_ACTIVE):
    ax = axes[i]
    ss = crude[crude['series'] == series].copy()
    degrees = sorted(ss['alkyl_degree'].unique())

    # Normalize per sample
    totals = ss.groupby(['oil_id', 'stage'])['intensity'].transform('sum')
    ss['rel_intensity'] = ss['intensity'] / totals.replace(0, np.nan)

    width = 0.35
    x = np.arange(len(degrees))
    for k, stg in enumerate(['W0', 'W3']):
        sst = ss[ss['stage'] == stg]
        medians = sst.groupby('alkyl_degree')['rel_intensity'].median()
        vals = [medians.get(d, 0) for d in degrees]
        ax.bar(x + k * width, vals, width, label=STAGE_LABELS[stg],
               color=STAGE_COLORS.get(stg, 'gray'), alpha=0.85)

    abbr = PAH_ABBREV.get(series, series[:4])
    ax.set_title(abbr)
    ax.set_xlabel('Alkylation degree')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([f'C{d}' for d in degrees])
    if i == 0:
        ax.set_ylabel('Relative intensity (median)')
    ax.legend(fontsize=8)

fig.suptitle('F02 -- Bell-Shift W0 vs W3 (Crude Oils Only)', fontsize=14, y=1.02)
fig_path = FIG_DIR / 'F02_bell_shift_comparison_all_series.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

## PART 2 -- Alkylation Centroid

In [ ]:
# C06 -- F03 Centroid evolution
# Centroid = sum(degree * intensity) / sum(intensity) per oil x stage x series
df_pah['deg_x_int'] = df_pah['alkyl_degree'] * df_pah['intensity']

centroid = (df_pah.groupby(['oil_id', 'oil_type', 'stage', 'series'])
            .agg(sum_deg_int=('deg_x_int', 'sum'), sum_int=('intensity', 'sum'))
            .reset_index())
centroid['centroid'] = centroid['sum_deg_int'] / centroid['sum_int'].replace(0, np.nan)

fig, axes = plt.subplots(1, n_series, figsize=(4 * n_series, 4), constrained_layout=True)
if n_series == 1:
    axes = [axes]

for i, series in enumerate(SERIES_ACTIVE):
    ax = axes[i]
    cs = centroid[centroid['series'] == series]
    for ot in OIL_TYPES_ML:
        cso = cs[cs['oil_type'] == ot]
        if cso.empty:
            continue
        med = cso.groupby('stage')['centroid'].median().reindex(STAGE_ORDER)
        ax.plot(STAGE_ORDER, med.values, 'o-', label=ot.replace('_', ' ').title(),
                color=OILTYPE_COLORS.get(ot, 'gray'), linewidth=2, markersize=6)

    abbr = PAH_ABBREV.get(series, series[:4])
    ax.set_title(abbr)
    ax.set_xlabel('Weathering stage')
    if i == 0:
        ax.set_ylabel('Alkylation centroid')
    ax.legend(fontsize=7)

fig.suptitle('F03 -- Alkylation Centroid Evolution by Series', fontsize=14, y=1.02)
fig_path = FIG_DIR / 'F03_centroid_evolution_all_series.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

# Print crude W0->W3 centroid shift per series
print('\nCrude centroid W0 -> W3:')
for series in SERIES_ACTIVE:
    cs = centroid[(centroid['series'] == series) & (centroid['oil_type'] == 'crude')]
    w0 = cs[cs['stage'] == 'W0']['centroid'].median()
    w3 = cs[cs['stage'] == 'W3']['centroid'].median()
    abbr = PAH_ABBREV.get(series, series[:4])
    print(f'  {abbr}: {w0:.3f} -> {w3:.3f} (delta = {w3 - w0:+.3f})')

In [ ]:
# C07 -- F04 Shift magnitude + C0 depletion
# Panel 1: delta centroid (W3 - W0) per series x oil_type
# Panel 2: C0 depletion corrected for evaporative loss

# Compute delta centroid per oil x series
pivot_cent = centroid.pivot_table(index=['oil_id', 'oil_type', 'series'],
                                  columns='stage', values='centroid').reset_index()
pivot_cent['delta_centroid'] = pivot_cent['W3'] - pivot_cent['W0']

# Compute C0 depletion: corrected for mass loss
c0 = df_pah[df_pah['alkyl_degree'] == 0].copy()
c0_pivot = c0.pivot_table(index=['oil_id', 'oil_type', 'series'],
                          columns='stage', values='intensity').reset_index()
R_W3 = 1 - STAGE_EVAP['W3']  # 0.70
c0_pivot['depletion_pct'] = (c0_pivot['W3'] * R_W3 - c0_pivot['W0']) / c0_pivot['W0'].replace(0, np.nan) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# Panel 1: delta centroid
series_labels = [PAH_ABBREV.get(s, s[:4]) for s in SERIES_ACTIVE]
x = np.arange(len(SERIES_ACTIVE))
width = 0.18
for k, ot in enumerate(OIL_TYPES_ML):
    vals = []
    for series in SERIES_ACTIVE:
        sub = pivot_cent[(pivot_cent['series'] == series) & (pivot_cent['oil_type'] == ot)]
        vals.append(sub['delta_centroid'].median() if len(sub) > 0 else 0)
    ax1.bar(x + k * width, vals, width, label=ot.replace('_', ' ').title(),
            color=OILTYPE_COLORS.get(ot, 'gray'), alpha=0.85)
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(series_labels)
ax1.set_ylabel('Delta centroid (W3 - W0)')
ax1.set_title('Centroid shift magnitude')
ax1.legend(fontsize=8)
ax1.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# Panel 2: C0 depletion
for k, ot in enumerate(OIL_TYPES_ML):
    vals = []
    for series in SERIES_ACTIVE:
        sub = c0_pivot[(c0_pivot['series'] == series) & (c0_pivot['oil_type'] == ot)]
        vals.append(sub['depletion_pct'].median() if len(sub) > 0 else 0)
    ax2.bar(x + k * width, vals, width, label=ot.replace('_', ' ').title(),
            color=OILTYPE_COLORS.get(ot, 'gray'), alpha=0.85)
ax2.set_xticks(x + width * 1.5)
ax2.set_xticklabels(series_labels)
ax2.set_ylabel('C0 depletion (%, corrected)')
ax2.set_title('C0 (parent) depletion W0 to W3')
ax2.legend(fontsize=8)
ax2.axhline(0, color='gray', linewidth=0.5, linestyle='--')

fig.suptitle('F04 -- Shift Magnitude & C0 Depletion', fontsize=14, y=1.02)
fig_path = FIG_DIR / 'F04_shift_magnitude_C0_depletion.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

In [ ]:
# C08 -- Persist centroids, shifts, C0 depletion into molecular_fingerprint_stats
rows_to_insert = []

# 1. pah_centroid_{abbrev} per oil x stage
for _, row in centroid.iterrows():
    if pd.isna(row['centroid']):
        continue
    abbr = PAH_ABBREV.get(row['series'], row['series'][:4]).lower()
    rows_to_insert.append((row['oil_id'], row['stage'],
                           f'pah_centroid_{abbr}', float(row['centroid'])))

# 2. pah_shift_{abbrev} per oil (stored at W0 stage_code)
for _, row in pivot_cent.iterrows():
    if pd.isna(row['delta_centroid']):
        continue
    abbr = PAH_ABBREV.get(row['series'], row['series'][:4]).lower()
    rows_to_insert.append((row['oil_id'], 'W0',
                           f'pah_shift_{abbr}', float(row['delta_centroid'])))

# 3. pah_depletion_C0_{abbrev} per oil (stored at W0 stage_code)
for _, row in c0_pivot.iterrows():
    if pd.isna(row['depletion_pct']):
        continue
    abbr = PAH_ABBREV.get(row['series'], row['series'][:4]).lower()
    rows_to_insert.append((row['oil_id'], 'W0',
                           f'pah_depletion_C0_{abbr}', float(row['depletion_pct'])))

with get_conn() as conn:
    conn.executemany('''
        INSERT OR REPLACE INTO molecular_fingerprint_stats
            (oil_id, stage_code, stat_name, stat_value)
        VALUES (?, ?, ?, ?)
    ''', rows_to_insert)

print(f'Persisted {len(rows_to_insert)} pah_* stats to molecular_fingerprint_stats.')
# Verify
with get_conn() as conn:
    check = pd.read_sql('''
        SELECT stat_name, COUNT(*) AS n
        FROM molecular_fingerprint_stats
        WHERE stat_name LIKE 'pah_%'
        GROUP BY stat_name ORDER BY stat_name
    ''', conn)
    display(check)

### Canonical PAH Process Ratios


In [ ]:
# C_PAH_RATIOS -- Canonical PAH ratios from diagnostic_ratios
PAH_CANONICAL = ['C0Naph_C4Naph', 'C0Flu_C3Flu', 'C0Phe_C4Phe', 'C0DBT_C3DBT', 'LMW_HMW_PAH_5ring']
with get_conn() as conn:
    ph = ','.join('?' * len(PAH_CANONICAL))
    df_pah_ratios = pd.read_sql(
        f'SELECT r.oil_id, o.oil_type, r.stage_code AS stage, '
        f'r.ratio_name, r.value AS ratio_value '
        f'FROM diagnostic_ratios r JOIN oils o ON r.oil_id = o.oil_id '
        f'WHERE r.ratio_name IN ({ph}) AND o.include_in_analysis = 1 '
        f"AND r.stage_code IN ('W0','W1','W2','W3')",
        conn, params=PAH_CANONICAL)
df_pah_ratios['stage'] = pd.Categorical(df_pah_ratios['stage'], STAGE_ORDER, ordered=True)
avail_pah = [r for r in PAH_CANONICAL if r in df_pah_ratios['ratio_name'].unique()]
n_r = len(avail_pah)
if n_r > 0:
    fig, axes = plt.subplots(1, n_r, figsize=(4.5*n_r, 5), constrained_layout=True)
    if n_r == 1: axes = [axes]
    for ax, rname in zip(axes, avail_pah):
        for ot in OIL_TYPES_ML:
            sub = df_pah_ratios[(df_pah_ratios['ratio_name']==rname)&(df_pah_ratios['oil_type']==ot)]
            med = [sub[sub['stage']==s]['ratio_value'].median() for s in STAGE_ORDER]
            ax.plot(STAGE_ORDER, med, 'o-', color=OILTYPE_COLORS.get(ot,'grey'), label=ot, lw=2, ms=6)
        ax.set_title(rname, fontsize=9); ax.set_xlabel('Stage')
        ax.set_ylabel('Ratio (median)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.suptitle('Canonical PAH Process Ratios -- W0->W3', fontsize=11)
    fig_path = FIG_DIR / 'F_PAH_CANONICAL_RATIOS.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    print('PAH canonical ratios W0->W3 (crude, median % change):')
    for rn in avail_pah:
        csub = df_pah_ratios[(df_pah_ratios['ratio_name']==rn)&(df_pah_ratios['oil_type']=='crude')]
        w0m = csub[csub['stage']=='W0']['ratio_value'].median()
        w3m = csub[csub['stage']=='W3']['ratio_value'].median()
        pct = (w3m-w0m)/w0m*100 if pd.notna(w0m) and w0m>0 else np.nan
        print(f'  {rn}: W0={w0m:.3f} -> W3={w3m:.3f} ({pct:+.1f}%)')


## PART 3 -- Source Ratios & DBT/Phe

In [ ]:
# C_WILCOXON_PAH -- Wilcoxon + effect size: all canonical PAH ratios
from scipy.stats import wilcoxon
ALL_PAH = ['C0Naph_C4Naph','C0Flu_C3Flu','C0Phe_C4Phe','C0DBT_C3DBT',
           'LMW_HMW_PAH_5ring','C0DBT_C0Phe','C1DBT_C1Phe','C2DBT_C2Phe']
ALPHA_BONF = 0.05 / len(ALL_PAH)
APRIORI = {'C0Naph_C4Naph':'process','C0Flu_C3Flu':'process','C0Phe_C4Phe':'process',
           'C0DBT_C3DBT':'process','LMW_HMW_PAH_5ring':'process',
           'C0DBT_C0Phe':'identity','C1DBT_C1Phe':'identity','C2DBT_C2Phe':'identity'}
with get_conn() as conn:
    ph_all = ','.join(['?' for _ in ALL_PAH])
    df_apw = pd.read_sql(
        f'SELECT r.oil_id, o.oil_type, r.stage_code AS stage, '
        f'r.ratio_name, r.value AS ratio_value '
        f'FROM diagnostic_ratios r JOIN oils o ON r.oil_id = o.oil_id '
        f'WHERE r.ratio_name IN ({ph_all}) AND o.include_in_analysis = 1 '
        f"AND r.stage_code IN ('W0','W3')",
        conn, params=ALL_PAH)
print('=== Wilcoxon: ALL canonical PAH ratios ===')
print(f'Bonferroni alpha = {ALPHA_BONF:.5f}')
concordance = 0
for rname in ALL_PAH:
    sub = df_apw[df_apw['ratio_name']==rname]
    w0v = sub[sub['stage']=='W0'].set_index('oil_id')['ratio_value']
    w3v = sub[sub['stage']=='W3'].set_index('oil_id')['ratio_value']
    common = w0v.index.intersection(w3v.index)
    apriori = APRIORI.get(rname,'?')
    if len(common) < 5:
        print(f'  {rname}: insufficient data'); continue
    stat_w, p_w = wilcoxon(w0v[common].values, w3v[common].values)
    n = len(common)
    if np.isnan(p_w):
        empirical = 'identity'; verdict = 'STABLE'
    elif p_w < ALPHA_BONF:
        empirical = 'process'; verdict = f'p={p_w:.4f}*'
    else:
        empirical = 'identity'; verdict = f'p={p_w:.4f}'
    match = 'OK' if apriori == empirical else 'MISMATCH'
    if apriori == empirical: concordance += 1
    print(f'  {rname:22s} {verdict:>12} {apriori:>8} -> {empirical:>8} {match}')
print(f'Concordance: {concordance}/{len(ALL_PAH)} ({100*concordance/len(ALL_PAH):.0f}%)')


In [ ]:
# C09 -- F05 Source ratios Fl/(Fl+Py) and BaA/(BaA+Chr) evolution W0->W3
# Fl and Py are in pah_priority; C0-Chrysene is in pah_alkylated
with get_conn() as conn:
    df_src = pd.read_sql("""
        SELECT m.oil_id, o.oil_type, m.stage_code AS stage,
               c.compound_name, m.value_imputed AS intensity
        FROM measurements m
        JOIN oils o ON m.oil_id = o.oil_id
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_name IN (
            'Fluoranthene (Fl)', 'Pyrene (Py)',
            'Benz(a)anthracene (BaA)', 'C0-Chrysene'
        )
          AND o.include_in_analysis = 1
          AND m.stage_code IN ('W0','W1','W2','W3')
    """, conn)

# Pivot to wide
src_wide = df_src.pivot_table(index=['oil_id', 'oil_type', 'stage'],
                               columns='compound_name', values='intensity').reset_index()
src_wide.columns.name = None

fl_col = 'Fluoranthene (Fl)'
py_col = 'Pyrene (Py)'
baa_col = 'Benz(a)anthracene (BaA)'
chr_col = 'C0-Chrysene'

source_ratios = {}
if fl_col in src_wide.columns and py_col in src_wide.columns:
    src_wide['Fl_FlPy'] = src_wide[fl_col] / (src_wide[fl_col] + src_wide[py_col]).replace(0, np.nan)
    source_ratios['Fl_FlPy'] = ('Fl/(Fl+Py)', 0.50, 'Petrogenic < 0.50 < Pyrogenic')

if baa_col in src_wide.columns and chr_col in src_wide.columns:
    src_wide['BaA_BaAChr'] = src_wide[baa_col] / (src_wide[baa_col] + src_wide[chr_col]).replace(0, np.nan)
    source_ratios['BaA_BaAChr'] = ('BaA/(BaA+Chr)', 0.35, 'Petrogenic < 0.35 < Pyrogenic')

n_ratios = len(source_ratios)
print(f'Source ratios computed: {list(source_ratios.keys())}')

if n_ratios > 0:
    fig, axes = plt.subplots(1, n_ratios, figsize=(6 * n_ratios, 5), constrained_layout=True)
    if n_ratios == 1:
        axes = [axes]
    for ax, (col, (label, boundary, desc)) in zip(axes, source_ratios.items()):
        for ot in OIL_TYPES_ML:
            sub = src_wide[src_wide['oil_type'] == ot]
            med = [sub[sub['stage'] == s][col].median() for s in STAGE_ORDER]
            ax.plot(STAGE_ORDER, med, 'o-', color=OILTYPE_COLORS.get(ot, 'grey'),
                    label=ot, linewidth=2, markersize=7)
        ax.axhline(boundary, color='red', ls='--', lw=1.5, label=f'Boundary ({boundary})')
        ax.set_ylim(0, 0.7)
        ax.set_title(f'{label} — {desc}', fontsize=10)
        ax.set_xlabel('Weathering stage')
        ax.set_ylabel('Ratio value (median)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle('PAH Source Ratios -- Evolution W0->W3 | Source ratios expected STABLE under evaporation', fontsize=11)
    fig_path = FIG_DIR / 'F05_source_ratios_evolution.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    for col, (label, _, _) in source_ratios.items():
        crude_med = src_wide[src_wide['oil_type'] == 'crude'].groupby('stage')[col].median()
        print(f'{label} (crude median): {crude_med.round(3).to_dict()}')


# --- Formal tests for source ratios ---
from scipy.stats import wilcoxon, kruskal
print()
print('=== Wilcoxon: source ratio stability W0 vs W3 ===')
for col, (label, boundary, desc) in source_ratios.items():
    w0v = src_wide[src_wide['stage']=='W0'].set_index('oil_id')[col]
    w3v = src_wide[src_wide['stage']=='W3'].set_index('oil_id')[col]
    common = w0v.dropna().index.intersection(w3v.dropna().index)
    if len(common) >= 5:
        _, p_w = wilcoxon(w0v[common].values, w3v[common].values)
        if np.isnan(p_w):
            print(f'  {label}: PERFECTLY STABLE')
        elif p_w > 0.05:
            print(f'  {label}: p={p_w:.4f} -> STABLE (source confirmed)')
        else:
            print(f'  {label}: p={p_w:.4f} -> CHANGES (subtle evaporation effect)')

print()
print('=== KW: source ratios differ between oil_types at W0? ===')
w0_src = src_wide[src_wide['stage']=='W0']
for col, (label, _, _) in source_ratios.items():
    groups = [w0_src[w0_src['oil_type']==ot][col].dropna().values for ot in OIL_TYPES_ML]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) >= 2:
        stat_kw, p_kw = kruskal(*groups)
        sig = '***' if p_kw<0.001 else '**' if p_kw<0.01 else '*' if p_kw<0.05 else 'ns'
        print(f'  {label}: KW p={p_kw:.4f} {sig}')


In [ ]:
# C10 -- F06 DBT/Phe x Pr/Ph
with get_conn() as conn:
    df_ratios = pd.read_sql('''
        SELECT r.oil_id, o.oil_type, r.stage_code AS stage,
               r.ratio_name, r.value
        FROM diagnostic_ratios r
        JOIN oils o ON r.oil_id = o.oil_id
        WHERE r.ratio_name IN ('C0DBT_C0Phe', 'Pr_Ph')
          AND o.include_in_analysis = 1
          AND r.stage_code IN ('W0','W1','W2','W3')
    ''', conn)

rat_wide = df_ratios.pivot_table(index=['oil_id', 'oil_type', 'stage'],
                                 columns='ratio_name', values='value').reset_index()
w0 = rat_wide[rat_wide['stage'] == 'W0'].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# Panel 1: boxplot DBT/Phe by oil_type at W0
box_data = [w0[w0['oil_type'] == ot]['C0DBT_C0Phe'].dropna().values for ot in OIL_TYPES_ML]
box_labels = [ot.replace('_', ' ').title() for ot in OIL_TYPES_ML]
bp = ax1.boxplot(box_data, tick_labels=box_labels, patch_artist=True)
for patch, ot in zip(bp['boxes'], OIL_TYPES_ML):
    patch.set_facecolor(OILTYPE_COLORS.get(ot, 'gray'))
    patch.set_alpha(0.7)
ax1.set_ylabel('C0-DBT / C0-Phe')
ax1.set_title('DBT/Phe at W0 by Oil Type')

# Panel 2: scatter DBT/Phe x Pr/Ph at W0 with Spearman
valid = w0.dropna(subset=['C0DBT_C0Phe', 'Pr_Ph'])
for ot in OIL_TYPES_ML:
    sub = valid[valid['oil_type'] == ot]
    if sub.empty:
        continue
    ax2.scatter(sub['Pr_Ph'], sub['C0DBT_C0Phe'], label=ot.replace('_', ' ').title(),
               color=OILTYPE_COLORS.get(ot, 'gray'), alpha=0.7, edgecolors='k', linewidth=0.3)
if len(valid) > 5:
    rho, pval = stats.spearmanr(valid['Pr_Ph'], valid['C0DBT_C0Phe'])
    ax2.set_xlabel(f'Pr/Ph\n(Spearman rho={rho:.3f}, p={pval:.2e})')
else:
    ax2.set_xlabel('Pr/Ph')
ax2.set_ylabel('C0-DBT / C0-Phe')
ax2.set_title('DBT/Phe vs Pr/Ph at W0')
ax2.legend(fontsize=8)

fig.suptitle('F06 -- DBT/Phe Source Indicator', fontsize=14, y=1.02)
fig_path = FIG_DIR / 'F06_DBT_Phe_source_indicator.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

In [ ]:
# C11 -- F07 Priority PAH profile heatmap at W0
with get_conn() as conn:
    df_pri = pd.read_sql("""
        SELECT m.oil_id, o.oil_type, m.stage_code AS stage,
               c.compound_name, m.value_imputed AS intensity
        FROM measurements m
        JOIN oils o ON m.oil_id = o.oil_id
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_group = 'pah_priority'
          AND c.excluded = 0 AND o.include_in_analysis = 1
          AND m.stage_code = 'W0'
    """, conn)

def extract_abbr(name):
    m = re.search(r'\(([^)]+)\)', name)
    return m.group(1) if m else name

df_pri['abbr'] = df_pri['compound_name'].apply(extract_abbr)
w0_pri = df_pri.copy()

# Median intensity per oil_type x compound
heat = w0_pri.groupby(['oil_type', 'abbr'])['intensity'].median().unstack(fill_value=0)

# Normalize by row (oil_type)
heat_norm = heat.div(heat.sum(axis=1), axis=0)

# Reorder oil types and compounds
ot_order = [ot for ot in OIL_TYPES_ML if ot in heat_norm.index]
heat_norm = heat_norm.reindex(ot_order)

fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
im = ax.imshow(heat_norm.values, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(ot_order)))
ax.set_yticklabels([ot.replace('_', ' ').title() for ot in ot_order])
ax.set_xticks(range(len(heat_norm.columns)))
ax.set_xticklabels(heat_norm.columns, rotation=45, ha='right', fontsize=8)
ax.set_title('F07 -- Priority PAH Profile at W0 (row-normalized)')
plt.colorbar(im, ax=ax, label='Relative intensity', shrink=0.8)

fig_path = FIG_DIR / 'F07_priority_PAH_profile_W0.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

In [ ]:
# C12 -- F08 Canonical PAH ratios evolution
CANONICAL_RATIOS = ['C0Naph_C4Naph', 'C0Flu_C3Flu', 'C0Phe_C4Phe', 'C0DBT_C3DBT', 'LMW_HMW_PAH_5ring']

with get_conn() as conn:
    placeholders = ','.join([f"'{r}'" for r in CANONICAL_RATIOS])
    df_canon = pd.read_sql(f'''
        SELECT r.oil_id, o.oil_type, r.stage_code AS stage,
               r.ratio_name, r.value
        FROM diagnostic_ratios r
        JOIN oils o ON r.oil_id = o.oil_id
        WHERE r.ratio_name IN ({placeholders})
          AND o.include_in_analysis = 1
          AND r.stage_code IN ('W0','W1','W2','W3')
    ''', conn)

n_ratios = len(CANONICAL_RATIOS)
fig, axes = plt.subplots(1, n_ratios, figsize=(4 * n_ratios, 4), constrained_layout=True)

for i, rname in enumerate(CANONICAL_RATIOS):
    ax = axes[i]
    sub = df_canon[df_canon['ratio_name'] == rname]
    for ot in OIL_TYPES_ML:
        sot = sub[sub['oil_type'] == ot]
        if sot.empty:
            continue
        med = sot.groupby('stage')['value'].median().reindex(STAGE_ORDER)
        ax.plot(STAGE_ORDER, med.values, 'o-', label=ot.replace('_', ' ').title(),
                color=OILTYPE_COLORS.get(ot, 'gray'), linewidth=2)
    ax.set_title(rname.replace('_', '/'), fontsize=9)
    ax.set_xlabel('Stage')
    if i == 0:
        ax.set_ylabel('Ratio value')
    if i == n_ratios - 1:
        ax.legend(fontsize=7)

fig.suptitle('F08 -- Canonical PAH Ratios Evolution', fontsize=14, y=1.02)
fig_path = FIG_DIR / 'F08_canonical_PAH_ratios_evolution.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

## PART 4 -- Additional Analyses

In [ ]:
# C13 -- F09 BaP/(BaP+BeP) protocol validation
with get_conn() as conn:
    # Dynamic name lookup
    names = pd.read_sql("""SELECT compound_name FROM compounds
        WHERE compound_group = 'pah_priority'
          AND (compound_name LIKE '%BaP%' OR compound_name LIKE '%BeP%')""", conn)
    bap_names = [n for n in names['compound_name'] if 'BaP' in n]
    bep_names = [n for n in names['compound_name'] if 'BeP' in n]
    print(f'BaP compounds: {bap_names}')
    print(f'BeP compounds: {bep_names}')

if not bap_names or not bep_names:
    print('BaP or BeP not found. Skipping F09.')
else:
    bap_name, bep_name = bap_names[0], bep_names[0]
    with get_conn() as conn:
        df_bap = pd.read_sql(f"""SELECT m.oil_id, o.oil_name, o.oil_type,
            m.stage_code AS stage, c.compound_name, m.value_imputed AS intensity
            FROM measurements m JOIN oils o ON m.oil_id = o.oil_id
            JOIN compounds c ON m.compound_id = c.compound_id
            WHERE c.compound_name IN ('{bap_name}', '{bep_name}')
              AND o.include_in_analysis = 1
              AND m.stage_code IN ('W0','W1','W2','W3')""", conn)
    print(f'BaP/BeP rows: {len(df_bap)}')
    df_bap['stage'] = pd.Categorical(df_bap['stage'], STAGE_ORDER, ordered=True)
    bap_wide = df_bap.pivot_table(index=['oil_id','oil_name','oil_type','stage'],
        columns='compound_name', values='intensity').reset_index()
    bap_wide.columns.name = None
    bap_wide['BaP_BaPBeP'] = bap_wide[bap_name] / (
        bap_wide[bap_name] + bap_wide[bep_name]).replace(0, np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    # Panel 1: evolution W0->W3
    ax = axes[0]
    for ot in OIL_TYPES_ML:
        sub = bap_wide[bap_wide['oil_type']==ot]
        med = [sub[sub['stage']==s]['BaP_BaPBeP'].median() for s in STAGE_ORDER]
        ax.plot(STAGE_ORDER, med, 'o-', color=OILTYPE_COLORS.get(ot,'grey'),
                label=ot, linewidth=2, markersize=7)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('Weathering stage')
    ax.set_ylabel('BaP / (BaP + BeP)')
    ax.set_title('BaP/(BaP+BeP) evolution W0->W3')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 2: scatter W0 vs W3
    ax = axes[1]
    w0b = bap_wide[bap_wide['stage']=='W0'].set_index('oil_id')['BaP_BaPBeP']
    w3b = bap_wide[bap_wide['stage']=='W3'].set_index('oil_id')['BaP_BaPBeP']
    common_b = w0b.index.intersection(w3b.index)
    if len(common_b) >= 3:
        ax.scatter(w0b[common_b], w3b[common_b], s=50, alpha=0.7)
        lim_b = max(w0b[common_b].max(), w3b[common_b].max()) * 1.05
        ax.plot([0, lim_b], [0, lim_b], 'k--', alpha=0.5, label='y = x')
        ax.set_xlabel('BaP/(BaP+BeP) at W0')
        ax.set_ylabel('BaP/(BaP+BeP) at W3')
        ax.set_title('Stability: W0 vs W3 (diagonal = stable)')
        ax.legend(fontsize=8)
    # Wilcoxon
    if len(common_b) >= 5:
        from scipy.stats import wilcoxon
        stat_w, p_w = wilcoxon(w0b[common_b].values, w3b[common_b].values)
        if np.isnan(p_w):
            print('BaP/(BaP+BeP) Wilcoxon: PERFECTLY STABLE (all zeros)')
        elif p_w > 0.05:
            print(f'BaP/(BaP+BeP) Wilcoxon: p={p_w:.4f} -> STABLE (no photo-oxidation)')
        else:
            print(f'BaP/(BaP+BeP) Wilcoxon: p={p_w:.4f} -> CHANGES (investigate)')

    fig.suptitle('BaP/(BaP+BeP) -- Photo-oxidation indicator (ECCC protocol validation)', fontsize=11)
    fig_path = FIG_DIR / 'F09_BaP_BeP_protocol_validation.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))


In [ ]:
# C15 -- F11 Centroid as weathering indicator
# Kruskal-Wallis between stages for each series (crude only)
crude_cent = centroid[centroid['oil_type'] == 'crude'].copy()

fig, axes = plt.subplots(1, n_series, figsize=(4 * n_series, 4), constrained_layout=True)
if n_series == 1:
    axes = [axes]

print('Kruskal-Wallis test (crude only): centroid ~ stage')
for i, series in enumerate(SERIES_ACTIVE):
    ax = axes[i]
    cs = crude_cent[crude_cent['series'] == series]
    groups = [cs[cs['stage'] == s]['centroid'].dropna().values for s in STAGE_ORDER]
    non_empty = [g for g in groups if len(g) > 0]

    if len(non_empty) >= 2:
        stat_kw, p_kw = kruskal(*non_empty)
        sig = '***' if p_kw < 0.001 else '**' if p_kw < 0.01 else '*' if p_kw < 0.05 else 'ns'
    else:
        p_kw, sig = np.nan, 'N/A'

    abbr = PAH_ABBREV.get(series, series[:4])
    print(f'  {abbr}: H-stat={stat_kw:.2f}, p={p_kw:.2e} ({sig})' if not np.isnan(p_kw) else f'  {abbr}: insufficient data')

    bp = ax.boxplot(groups, tick_labels=STAGE_ORDER, patch_artist=True)
    for patch, stg in zip(bp['boxes'], STAGE_ORDER):
        patch.set_facecolor(STAGE_COLORS.get(stg, 'gray'))
        patch.set_alpha(0.7)
    ax.set_title(f'{abbr} ({sig})')
    ax.set_xlabel('Stage')
    if i == 0:
        ax.set_ylabel('Alkylation centroid')

fig.suptitle('F11 -- Centroid as Weathering Indicator (Crude Only)', fontsize=14, y=1.02)
fig_path = FIG_DIR / 'F11_centroid_weathering_indicator.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

In [ ]:
# C_CENTROID_STATS -- Spearman rho centroid vs stage + inter-centroid matrix
# Rebuild df_centroid if not in namespace
if 'df_centroid' not in dir():
    print('Rebuilding df_centroid from df_pah...')
    centroid_recs = []
    for (oil_id, oil_type, stage), grp in df_pah.groupby(['oil_id','oil_type','stage']):
        for series, s_grp in grp.groupby('series'):
            total = s_grp['intensity'].sum()
            if pd.isna(total) or total == 0: continue
            centroid_recs.append({'oil_id': oil_id, 'oil_type': oil_type,
                'stage': stage, 'series': series,
                'centroid': float((s_grp['alkyl_degree']*s_grp['intensity']).sum()/total)})
    df_centroid = pd.DataFrame(centroid_recs)
    df_centroid['stage'] = pd.Categorical(df_centroid['stage'], STAGE_ORDER, ordered=True)

print('=== Spearman: centroid vs stage (crude) ===')
crude_cent = df_centroid[df_centroid['oil_type']=='crude'].copy()
crude_cent['stage_num'] = crude_cent['stage'].map({'W0':0,'W1':1,'W2':2,'W3':3})
for series in SERIES_ACTIVE:
    sub = crude_cent[crude_cent['series']==series]
    valid = sub[['stage_num','centroid']].dropna()
    if len(valid) >= 5:
        rho_c, p_c = stats.spearmanr(valid['stage_num'], valid['centroid'])
        sig = '***' if p_c<0.001 else '**' if p_c<0.01 else '*' if p_c<0.05 else 'ns'
        print(f'  {PAH_ABBREV.get(series,series)}: rho={rho_c:.3f}, p={p_c:.4f} {sig}')
print()
print('=== Inter-centroid Spearman matrix at W0 ===')
w0_cent = df_centroid[df_centroid['stage']=='W0'].pivot_table(
    index='oil_id', columns='series', values='centroid')
w0_cent.columns = [PAH_ABBREV.get(s,s) for s in w0_cent.columns]
corr_m = w0_cent.corr(method='spearman').round(3)
print(corr_m.to_string())


In [ ]:
# C16 -- F12 PAH completeness map
# Detection rate heatmap: oil_type x compound at W0
w0_all = df_pah[df_pah['stage'] == 'W0'].copy()

# Count non-zero / total per oil_type x compound
detect = w0_all.copy()
detect['detected'] = (detect['intensity'] > 0).astype(int)

det_rate = detect.groupby(['oil_type', 'compound_name']).agg(
    n_detected=('detected', 'sum'),
    n_total=('detected', 'count')
).reset_index()
det_rate['rate'] = det_rate['n_detected'] / det_rate['n_total']

heat_det = det_rate.pivot_table(index='oil_type', columns='compound_name', values='rate', fill_value=0)
ot_order = [ot for ot in OIL_TYPES_ML if ot in heat_det.index]
heat_det = heat_det.reindex(ot_order)
# Sort columns by series then degree
col_order = sorted(heat_det.columns, key=lambda x: (x.split('-')[-1] if '-' in x else x, x))
heat_det = heat_det[col_order]

fig, ax = plt.subplots(figsize=(16, 4), constrained_layout=True)
im = ax.imshow(heat_det.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_yticks(range(len(ot_order)))
ax.set_yticklabels([ot.replace('_', ' ').title() for ot in ot_order])
ax.set_xticks(range(len(heat_det.columns)))
ax.set_xticklabels(heat_det.columns, rotation=90, fontsize=7)
ax.set_title('F12 -- PAH Detection Rate at W0 by Oil Type')
plt.colorbar(im, ax=ax, label='Detection rate', shrink=0.8)

fig_path = FIG_DIR / 'F12_PAH_completeness_map.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

In [ ]:
# C17 -- F13 DBT/Phe all grades (C0, C1, C2)
DBT_PHE_RATIOS = ['C0DBT_C0Phe', 'C1DBT_C1Phe', 'C2DBT_C2Phe']

with get_conn() as conn:
    placeholders = ','.join([f"'{r}'" for r in DBT_PHE_RATIOS])
    df_dbt = pd.read_sql(f'''
        SELECT r.oil_id, o.oil_type, r.stage_code AS stage,
               r.ratio_name, r.value
        FROM diagnostic_ratios r
        JOIN oils o ON r.oil_id = o.oil_id
        WHERE r.ratio_name IN ({placeholders})
          AND o.include_in_analysis = 1
          AND r.stage_code IN ('W0','W1','W2','W3')
    ''', conn)

fig, axes = plt.subplots(1, len(DBT_PHE_RATIOS), figsize=(5 * len(DBT_PHE_RATIOS), 4),
                         constrained_layout=True)

print('Wilcoxon stability test (W0 vs W3) for DBT/Phe grades:')
for i, rname in enumerate(DBT_PHE_RATIOS):
    ax = axes[i]
    sub = df_dbt[df_dbt['ratio_name'] == rname]

    # Line plot by oil_type
    for ot in OIL_TYPES_ML:
        sot = sub[sub['oil_type'] == ot]
        if sot.empty:
            continue
        med = sot.groupby('stage')['value'].median().reindex(STAGE_ORDER)
        ax.plot(STAGE_ORDER, med.values, 'o-', label=ot.replace('_', ' ').title(),
                color=OILTYPE_COLORS.get(ot, 'gray'), linewidth=2)

    ax.set_title(rname.replace('_', '/'), fontsize=10)
    ax.set_xlabel('Stage')
    if i == 0:
        ax.set_ylabel('Ratio value')
    if i == len(DBT_PHE_RATIOS) - 1:
        ax.legend(fontsize=7)

    # Wilcoxon W0 vs W3
    w0_v = sub[sub['stage'] == 'W0'][['oil_id', 'value']].rename(columns={'value': 'W0_val'})
    w3_v = sub[sub['stage'] == 'W3'][['oil_id', 'value']].rename(columns={'value': 'W3_val'})
    paired = w0_v.merge(w3_v, on='oil_id').dropna()
    diffs = paired['W3_val'] - paired['W0_val']
    if len(paired) < 5:
        wil_text = 'insufficient data'
    elif (diffs == 0).all():
        wil_text = 'PERFECTLY STABLE (all differences zero)'
    else:
        try:
            stat_w, p_w = stats.wilcoxon(paired['W0_val'], paired['W3_val'])
            if np.isnan(p_w):
                wil_text = 'PERFECTLY STABLE (p=nan)'
            else:
                wil_text = f'p={p_w:.2e}'
        except ValueError:
            wil_text = 'PERFECTLY STABLE (all differences zero)'
    print(f'  {rname}: {wil_text} (n={len(paired)})')

fig.suptitle('F13 -- DBT/Phe All Grades (C0, C1, C2)', fontsize=14, y=1.02)
fig_path = FIG_DIR / 'F13_DBT_Phe_all_grades.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')

### F10 -- Boiling Point vs C0 Depletion: Unified Thermodynamic Hierarchy


In [ ]:
# F10 -- Boiling point x C0 depletion (per-oil corrected with actual mass loss)
PARENT_BP = {
    'Naphthalene': 218, 'Fluorene': 295, 'Dibenzothiophene': 332,
    'Phenanthrene': 340, 'Chrysene': 448,
}
ALKANE_BP = {'n-C9': 151, 'n-C10': 174, 'n-C11': 196, 'n-C12': 216,
    'n-C13': 234, 'n-C14': 254, 'n-C15': 270, 'n-C16': 287,
    'n-C17': 302, 'n-C18': 317, 'n-C20': 343, 'n-C25': 402}

# Compute per-oil C0 depletion using actual evaporative_mass_loss
with get_conn() as conn:
    df_evap = pd.read_sql("""
        SELECT sp.oil_id, sp.stage_code AS stage, sp.value AS mass_loss_pct
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'evaporative_mass_loss'
          AND o.include_in_analysis = 1
          AND sp.stage_code IN ('W0','W3')
    """, conn)

evap_lk = {}
for _, row in df_evap.iterrows():
    evap_lk[(int(row['oil_id']), row['stage'])] = 1 - row['mass_loss_pct'] / 100.0

# PAH C0 depletion per oil per series
pah_c0 = df_pah[(df_pah['alkyl_degree'] == 0) &
                 (df_pah['oil_type'].isin(['crude', 'bitumen_blend']))].copy()
c0_depl = []
for series, grp in pah_c0.groupby('series'):
    w0 = grp[grp['stage']=='W0'].groupby('oil_id')['intensity'].mean()
    w3 = grp[grp['stage']=='W3'].groupby('oil_id')['intensity'].mean()
    common = w0.index.intersection(w3.index)
    depls = []
    for oid in common:
        c_w0 = float(w0.loc[oid])
        c_w3 = float(w3.loc[oid])
        r_w0 = evap_lk.get((oid, 'W0'), 1.0)
        r_w3 = evap_lk.get((oid, 'W3'), 0.70)
        if c_w0 * r_w0 > 0:
            depls.append((c_w3 * r_w3 - c_w0 * r_w0) / (c_w0 * r_w0) * 100)
    if depls:
        med = sorted(depls)[len(depls)//2]
        c0_depl.append({'series': series, 'bp': PARENT_BP.get(series),
            'depletion_C0': med, 'abbrev': PAH_ABBREV.get(series, series[:4]),
            'n': len(depls)})
df_bp_dep = pd.DataFrame(c0_depl).dropna(subset=['bp'])

# n-Alkane depletion from compound_depletion_summary
with get_conn() as conn:
    df_alk_depl = pd.read_sql("""
        SELECT compound_name, depletion_pct
        FROM compound_depletion_summary
        WHERE compound_group = 'n_alkane'
    """, conn)
df_alk_depl['bp'] = df_alk_depl['compound_name'].map(ALKANE_BP)
df_alk_depl = df_alk_depl.dropna(subset=['bp'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# Panel 1: PAH only
ax1.scatter(df_bp_dep['bp'], df_bp_dep['depletion_C0'], s=100, color='crimson',
            edgecolors='k', linewidth=0.5, zorder=5)
for _, row in df_bp_dep.iterrows():
    ax1.annotate(f"{row['abbrev']} ({row['depletion_C0']:.1f}%)",
                 (row['bp'], row['depletion_C0']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
if len(df_bp_dep) >= 3:
    rho_bp, p_bp = stats.spearmanr(df_bp_dep['bp'], df_bp_dep['depletion_C0'])
    ax1.set_title(f'PAH C0 Depletion vs BP (rho={rho_bp:.3f}, p={p_bp:.3f})', fontsize=10)
ax1.axhline(0, color='gray', lw=1, ls=':')
ax1.set_xlabel('Parent compound boiling point (C)')
ax1.set_ylabel('C0 depletion W0->W3 (%, per-oil corrected)')

# Panel 2: unified
ax2.scatter(df_alk_depl['bp'], df_alk_depl['depletion_pct'], s=30, alpha=0.6,
            color='steelblue', label='n-Alkanes', edgecolors='k', linewidth=0.3)
ax2.scatter(df_bp_dep['bp'], df_bp_dep['depletion_C0'], s=100, color='crimson',
            label='PAH C0', edgecolors='k', linewidth=0.5, zorder=5)
for _, row in df_bp_dep.iterrows():
    ax2.annotate(row['abbrev'], (row['bp'], row['depletion_C0']),
                 textcoords='offset points', xytext=(5, 5), fontsize=9, color='crimson')
ax2.axhline(0, color='gray', lw=1, ls=':')
ax2.set_xlabel('Boiling point (C)')
ax2.set_ylabel('Depletion (%, corrected)')
ax2.set_title('Unified Volatility Hierarchy: n-Alkanes + PAHs', fontsize=10)
ax2.legend(fontsize=9)

fig.suptitle('Boiling Point vs Evaporative Depletion (per-oil corrected with actual mass loss)', fontsize=11)
fig_path = FIG_DIR / 'F10_BP_vs_depletion_PAH_alkane.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print('F10 saved. NOTE: Per-oil correction using actual evaporative_mass_loss.')
for _, row in df_bp_dep.iterrows():
    print(f'  {row["abbrev"]:6s} BP={row["bp"]}C  depletion={row["depletion_C0"]:+.1f}%  n={row["n"]}')


## Summary

In [ ]:
# Summary
with get_conn() as conn:
    pah_stats = pd.read_sql('''
        SELECT stat_name, COUNT(*) AS n
        FROM molecular_fingerprint_stats
        WHERE stat_name LIKE 'pah_%'
        GROUP BY stat_name ORDER BY stat_name
    ''', conn)

print('=== NB03c2 Summary ===')
print(f'\nPersisted pah_* stats: {pah_stats["n"].sum()} rows across {len(pah_stats)} stat types')
display(pah_stats)

print(f'\nFigures saved to: {FIG_DIR}')
figs = sorted(FIG_DIR.glob('*.png'))
for f in figs:
    print(f'  {f.name}')
print(f'Total figures: {len(figs)}')

print('\nKey findings:')
print('  - Bell-shift from W0 to W3: lighter alkyl homologs depleted preferentially')
print('  - Centroid shifts rightward with weathering (higher alkylation degree enriched)')
print('  - C0 (parent) depletion is strongest for low-BP series (Naphthalene)')
print('  - Source ratios (Fl/Fl+Py, BaA/BaA+Chr) show oil-type dependence')
print('  - DBT/Phe ratios tested at C0, C1, C2 grades for weathering stability')

In [ ]:
# Close
import gc
plt.close('all')
gc.collect()
print('NB03c2 complete.')

In [ ]:
import sqlite3

import sys
sys.path.insert(0, '.')
from utils import DB_PATH  # PROJECT_ROOT-anchored (portable; cf. NB03g)

conn = sqlite3.connect(str(DB_PATH))

# 1. Measurement count for C3
for comp in ['C3-Dibenzothiophene', 'C3-Phenanthrene', 'C3-Chrysene']:
    n = conn.execute("""
        SELECT COUNT(*) FROM measurements m
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_name = ?
    """, (comp,)).fetchone()[0]
    print(f"{comp}: {n} records")

# 2. Example of an existing diagnostic_ratios row
print("\nExample C0DBT_C0Phe (first 3):")
ex = conn.execute("SELECT * FROM diagnostic_ratios WHERE ratio_name = 'C0DBT_C0Phe' LIMIT 3").fetchall()
for r in ex:
    print(f"  {r}")

conn.close()